In [4]:
import numpy as np

class PolicyNet2Layer:
    def __init__(self, input_dim=3, hidden_dim=8, output_dim=4):
        # Initialize weights
        self.W1 = np.random.randn(hidden_dim, input_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(output_dim, hidden_dim) * 0.1
        self.b2 = np.zeros(output_dim)

    def forward(self, state):
        self.state = state
        self.z1 = self.W1 @ state + self.b1
        self.a1 = np.tanh(self.z1)  # hidden layer activation
        self.output = self.W2 @ self.a1 + self.b2
        self.mu = self.output[:2]
        self.log_sigma = self.output[2:]
        self.sigma = np.exp(self.log_sigma)
        return self.mu, self.sigma

    def sample_action(self):
        action = np.random.normal(self.mu, self.sigma)
        return action

    def log_prob(self, action):
        var = self.sigma ** 2
        logp = -0.5 * ((action - self.mu)**2 / var + np.log(2 * np.pi * var))
        return np.sum(logp)  # sum log-probs across 2 actions

    def compute_gradients(self, action, return_R):
        # Derivative of log-prob w.r.t. mu and sigma
        delta = (action - self.mu) / (self.sigma**2)
        dlogp_dmu = delta
        dlogp_dlogsigma = -1 + (action - self.mu)**2 / (self.sigma**2)

        grad_output = np.concatenate([dlogp_dmu, dlogp_dlogsigma]) * return_R

        # Backprop through output layer
        dW2 = np.outer(grad_output, self.a1)
        db2 = grad_output
        da1 = self.W2.T @ grad_output
        dz1 = da1 * (1 - np.tanh(self.z1)**2)  # tanh grad
        dW1 = np.outer(dz1, self.state)
        db1 = dz1

        return dW1, db1, dW2, db2

    def update(self, grads, lr=1e-2):
        dW1, db1, dW2, db2 = grads
        self.W1 += lr * dW1
        self.b1 += lr * db1
        self.W2 += lr * dW2
        self.b2 += lr * db2

# === Example usage ===
np.random.seed(42)
policy = PolicyNet2Layer()
policy.W1

array([[ 0.04967142, -0.01382643,  0.06476885],
       [ 0.15230299, -0.02341534, -0.0234137 ],
       [ 0.15792128,  0.07674347, -0.04694744],
       [ 0.054256  , -0.04634177, -0.04657298],
       [ 0.02419623, -0.19132802, -0.17249178],
       [-0.05622875, -0.10128311,  0.03142473],
       [-0.09080241, -0.14123037,  0.14656488],
       [-0.02257763,  0.00675282, -0.14247482]])

In [29]:
state = np.array([0.5, -1.0, 2.0])
policy.mu, policy.sigma = policy.forward(state)
action = policy.sample_action()
logp = policy.log_prob(action)

# Suppose reward return R = 2.0
grads = policy.compute_gradients(action, return_R=2.0)
policy.update(grads)

In [30]:
action.tolist()

[-0.3646880561910315, -0.13940948998923974]

In [12]:
action

array([0.21922912, 0.86021863])

In [33]:
for x in reversed([0,10,2,3,4]):
    print(x)

4
3
2
10
0


In [37]:
data = np.load("drone_episode_13000.npz")
data['W1']

array([[-6.23191241e+05,  1.12090337e+04],
       [-8.16103215e+05, -1.76310907e+06],
       [-4.91734200e+07,  1.15966211e+08],
       [-3.04892554e+08, -4.01312218e+08],
       [-6.70642059e+07, -6.23196370e+07],
       [-6.12909526e+06, -1.69643328e+07],
       [ 1.14374667e+07,  7.60023359e+07],
       [-5.98380146e+06, -5.28356884e+06]])